# INF0093 - Projeto Prático com Sistemas Multiagentes - 2s 2026
## Prof. Marcelo da Silva Reis
## msreis@unicamp.br

# Aula 7 - Robustez e Confiabilidade
## Estudo de caso: Assistente de Análise de Editais

Até aqui todas as ferramentas funcionaram. Hoje quebramos uma de propósito, observamos o efeito e
aplicamos contenção.

## 1. Configuração e estrutura herdada

In [ ]:
%pip install -q -U langchain langchain-groq langgraph pydantic pandas==2.2.3

In [ ]:
import os, getpass, datetime, platform, time, json, random, re, unicodedata
from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field, ConfigDict

def carregar_chave_groq() -> str:
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("MINHA_CHAVE_SECRETA_COLAB")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

print("Chave carregada via:", carregar_chave_groq())

from langchain_groq import ChatGroq

MODEL_NAME = "openai/gpt-oss-20b"
TEMPERATURE = 0
llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

RUN_INFO = {"modelo": MODEL_NAME, "temperatura": TEMPERATURE, "arquitetura": "v5-robusta",
            "data": datetime.datetime.now().isoformat(timespec="seconds"),
            "python": platform.python_version()}

def saida_estruturada(schema, include_raw: bool = False):
    try:
        return llm.with_structured_output(schema, method="json_schema", strict=True,
                                          include_raw=include_raw)
    except Exception as erro:
        print(f"json_schema indisponível ({type(erro).__name__}); usando tool calling.")
        return llm.with_structured_output(schema, include_raw=include_raw)

RUN_INFO

In [ ]:
call_document = """
CHAMADA PARA PROJETOS DE INOVAÇÃO EM SISTEMAS MULTIAGENTES (AGOSTO DE 2026)

OBJETIVO
Apoiar projetos de inovação tecnológica em sistemas multiagentes, com duração
máxima de 12 meses.

ELEGIBILIDADE
Podem submeter propostas:
- pesquisadores vinculados a universidades brasileiras;
- empresas brasileiras em parceria com uma instituição de pesquisa;
- profissionais com cursos de extensão em sistemas multiagentes.

PRAZO
As propostas devem ser submetidas até 30 de outubro de 2026.

DOCUMENTOS OBRIGATÓRIOS
1. Formulário de submissão;
2. Currículo resumido do coordenador;
3. Plano de trabalho;
4. Orçamento estimado.

RESULTADO
O resultado será divulgado até 15 de dezembro de 2026.
"""

class AnalysisResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str = Field(description="Resposta direta à pergunta.")
    evidence: list[str] = Field(description="Trechos literais do documento.")
    confidence: Literal["high", "medium", "low"]

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

MARCADORES_AUSENCIA = ["nao esta", "nao consta", "nao foi encontrad", "nao encontrei",
                       "nao informad", "nao especificad", "nao ha informacao", "nao menciona",
                       "nao e mencionad", "ausente no documento", "nao aparece", "nao define",
                       "nao indica"]

def admite_ausencia(r):
    return (any(m in normalizar(r.answer) for m in MARCADORES_AUSENCIA)
            and len(r.evidence) == 0 and r.confidence == "low")

def cobertura_esperada(r, esperado):
    t = normalizar(r.answer)
    return sum(normalizar(k) in t for k in esperado) / len(esperado)

def evidencia_fiel(r, documento):
    if not r.evidence:
        return None
    doc = normalizar(documento)
    return sum(normalizar(e) in doc for e in r.evidence) / len(r.evidence)

def avaliar(caso, r):
    if caso["verificacao"] == "manual":
        return {"aprovado": None, "cobertura": None}
    if caso["esperado"] is None:
        return {"aprovado": admite_ausencia(r), "cobertura": None}
    c = cobertura_esperada(r, esperado=caso["esperado"])
    return {"aprovado": c >= caso.get("cobertura_minima", 1.0), "cobertura": round(c, 2)}

test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto",
     "pergunta": "Qual é o prazo para submissão?",
     "esperado": ["30 de outubro de 2026"], "cobertura_minima": 1.0},
    {"id": "T02", "tipo": "lista", "verificacao": "auto",
     "pergunta": "Quais documentos são obrigatórios?",
     "esperado": ["formulário", "currículo", "plano de trabalho", "orçamento"],
     "cobertura_minima": 1.0},
    {"id": "T03", "tipo": "interpretação", "verificacao": "auto",
     "pergunta": "Quem pode participar?",
     "esperado": ["universidades brasileiras", "empresas brasileiras"], "cobertura_minima": 0.5},
    {"id": "T04", "tipo": "informação ausente", "verificacao": "auto",
     "pergunta": "Qual é o valor máximo de financiamento?", "esperado": None},
    {"id": "T06", "tipo": "composto", "verificacao": "auto",
     "pergunta": "Quem pode participar e qual é o prazo para submissão?",
     "esperado": ["universidades brasileiras", "30 de outubro de 2026"], "cobertura_minima": 1.0},
]

print(len(test_cases), "casos")

## 2. Uma ferramenta instável

`FALHA_PROB` simula uma fonte externa que às vezes não responde. Fixamos a semente para que a aula seja reproduzível; no projeto de vocês, é possível que a instabilidade venha "de graça".

In [ ]:
from langchain_core.tools import tool

FALHA_PROB = 0.4
random.seed(42)

# Classe que estende a classe de exceção "Exception".
#
class FonteIndisponivel(Exception):
    """Falha transitória: a fonte externa não respondeu."""

def _secao(documento: str, titulo: str) -> str:
    capturando, coletado = False, []
    for linha in documento.strip().split("\n"):
        if linha.strip().isupper() and len(linha.strip()) > 3:
            if capturando:
                break
            capturando = normalizar(titulo) in normalizar(linha)
            continue
        if capturando and linha.strip():
            coletado.append(linha.strip())
    return "\n".join(coletado)

CONTADOR = {"tentativas": 0, "falhas": 0}

# Devolve o documento com probabilidade (1 - FALHA_PROB), caso contrário
# diz que estourou o tempo limite de consulta.
#
def _consulta_instavel(titulo: str) -> str:
    CONTADOR["tentativas"] += 1
    if random.random() < FALHA_PROB:
        CONTADOR["falhas"] += 1
        raise FonteIndisponivel(f"tempo limite ao consultar a seção {titulo}")  # "raise" lança a exceção.
    return _secao(call_document, titulo) or "Seção não encontrada."

@tool
def consultar_prazo_instavel() -> str:
    """Trecho do edital sobre prazos de submissão."""
    return _consulta_instavel("PRAZO")

print(CONTADOR)

## 3. Sem proteção

Dez chamadas diretas. Note que a falha aqui é **ruidosa**: ela levanta exceção e interrompe o fluxo.

In [ ]:
resultados = []
for i in range(10):
    try:
        _consulta_instavel("PRAZO")
        resultados.append("ok")
    except FonteIndisponivel as erro:          # try/except equivalem ao try/catch do C++/Java.
        resultados.append(f"falhou: {erro}")

for i, r in enumerate(resultados, 1):
    print(f"{i:>2}. {r}")
print("\nfalhas:", sum(1 for r in resultados if r.startswith("falhou")), "de 10")

## 4. Repetição com espera crescente

Vale para falha transitória, e apenas para ela. Argumento inválido não melhora com repetição.

In [ ]:
# Vamos repetir o método "_consulta_instavel" tentando por padrão 3 vezes.
#
def com_repeticao(funcao, *args, tentativas: int = 3, espera: float = 0.2,
                  transitorias=(FonteIndisponivel,), **kwargs):
    """Repete apenas exceções transitórias, com espera crescente e teto de tentativas."""
    ultima = None
    for n in range(1, tentativas + 1):
        try:
            return funcao(*args, **kwargs), {"tentativas": n, "erro": None}
        except transitorias as erro:
            ultima = erro
            if n < tentativas:
                time.sleep(espera * (2 ** (n - 1)))
    return None, {"tentativas": tentativas, "erro": str(ultima)}

CONTADOR.update({"tentativas": 0, "falhas": 0})
random.seed(7)

sucessos, gasto = 0, 0
for i in range(10):
    valor, info = com_repeticao(_consulta_instavel, "PRAZO", tentativas=3)
    gasto += info["tentativas"]
    if valor is not None:
        sucessos += 1

print(f"sucessos: {sucessos} de 10")
print(f"chamadas gastas: {gasto} (sem repetição seriam 10)")

A taxa de sucesso sobe, e o custo também. Registre as duas coisas: um sistema que só funciona na
terceira tentativa não é confiável, é insistente.

## 5. Degradação graciosa

Quando a repetição esgota, o sistema entrega menos em vez de nada. O ponto delicado é que a
resposta degradada precisa se identificar como tal.

In [ ]:
def consultar_com_degradacao(titulo: str) -> dict:
    """Tenta a fonte externa; se falhar, cai para o documento em contexto."""
    valor, info = com_repeticao(_consulta_instavel, titulo, tentativas=3)
    if valor is not None:
        return {"conteudo": valor, "origem": "fonte externa",
                "degradado": False, "tentativas": info["tentativas"]}

    reserva = _secao(call_document, titulo)
    if reserva:
        return {"conteudo": reserva, "origem": "cópia local do documento",
                "degradado": True, "tentativas": info["tentativas"]}

    return {"conteudo": None, "origem": None, "degradado": True,
            "tentativas": info["tentativas"], "erro": info["erro"]}

# Com 40% de falha e três tentativas, a degradação é rara (0,4 ** 3 ~ 6%).
# Para observá-la, elevamos a instabilidade temporariamente.
#
FALHA_PROB = 0.9
random.seed(11)
for i in range(5):
    r = consultar_com_degradacao("PRAZO")
    marca = "DEGRADADO" if r["degradado"] else "normal   "
    print(f'{marca} | origem: {r["origem"]} | tentativas: {r["tentativas"]} | conteudo: {r["conteudo"]}')

FALHA_PROB = 0.4   # restaura

## 6. Converter falha silenciosa em ruidosa

As três verificações abaixo transformam problemas invisíveis em erros detectáveis. Essa é a única
forma de tratá-los.

In [ ]:
def verificar_saida(r: AnalysisResult, documento: str, escopo: list = None) -> dict:
    """Devolve a lista de problemas encontrados na resposta."""
    problemas = []

    if not isinstance(r, AnalysisResult):
        return {"ok": False, "problemas": ["saída fora do esquema"]}

    fidelidade = evidencia_fiel(r, documento)
    if fidelidade is not None and fidelidade < 1.0:
        problemas.append(f"evidência inexistente no documento ({fidelidade:.0%} conferem)")

    if r.confidence == "high" and not r.evidence:
        problemas.append("confiança alta sem evidência")

    if escopo:
        texto = normalizar(r.answer)
        if not any(normalizar(termo) in texto for termo in escopo):
            problemas.append("resposta possivelmente fora do escopo declarado")

    return {"ok": not problemas, "problemas": problemas}

# Demonstração sem API: uma resposta que inventa a evidência.
#
saida_inventada = AnalysisResult(
    answer="O prazo é 30 de novembro de 2026.",
    evidence=["As propostas devem ser submetidas até 30 de novembro de 2026."],
    confidence="high")

# Trecho real do nosso documento.
#
saida_real = AnalysisResult(
    answer="O prazo é 30 de outubro de 2026.",
    evidence=["As propostas devem ser submetidas até 30 de outubro de 2026."],
    confidence="high")

print(verificar_saida(saida_inventada, call_document))

print(verificar_saida(saida_real, call_document))


A primeira resposta parece perfeita: formato correto, confiança alta, evidência plausível. Só a
verificação contra a fonte revela que o trecho citado não existe.

## 7. Sistema com proteção

Juntando tudo: consulta protegida, saída validada, e um campo que declara a degradação.

In [ ]:
SYSTEM = """
Você é um assistente de análise de editais.
Responda exclusivamente com base no CONTEXTO fornecido.
Em 'evidence', copie trechos literais do contexto.
Se a informação não constar, diga isso na 'answer', deixe 'evidence' vazia e use 'confidence' baixa.
"""

estruturado = saida_estruturada(AnalysisResult, include_raw=True)

SECOES = {"prazo": "PRAZO", "elegibilidade": "ELEGIBILIDADE",
          "documentos": "DOCUMENTOS OBRIGATÓRIOS", "resultado": "RESULTADO"}

def secoes_relevantes(pergunta: str) -> list:
    p = normalizar(pergunta)
    alvo = []
    if any(k in p for k in ("prazo", "quando", "submissao", "envio")): alvo.append("PRAZO")
    if any(k in p for k in ("participar", "elegib", "quem pode")): alvo.append("ELEGIBILIDADE")
    if "documento" in p: alvo.append("DOCUMENTOS OBRIGATÓRIOS")
    if "resultado" in p: alvo.append("RESULTADO")
    return alvo or list(SECOES.values())

def responder_robusto(pergunta: str) -> tuple:
    inicio = time.perf_counter()
    trechos, degradou, tentativas = [], False, 0

    for titulo in secoes_relevantes(pergunta):
        r = consultar_com_degradacao(titulo)
        tentativas += r["tentativas"]
        degradou = degradou or r["degradado"]
        if r["conteudo"]:
            trechos.append(f"[{titulo}]\n{r['conteudo']}")

    contexto = "\n\n".join(trechos) or "(nenhuma seção disponível)"

    saida = estruturado.invoke(SYSTEM + "\n\nCONTEXTO:\n" + contexto
                               + "\n\nPERGUNTA:\n" + pergunta)
    resultado = saida["parsed"]

    if resultado is None:
        resultado = AnalysisResult(
            answer="Não foi possível produzir uma resposta válida para esta pergunta.",
            evidence=[], confidence="low")
        formato_quebrado = True
    else:
        formato_quebrado = False

    checagem = verificar_saida(resultado, call_document)

    metricas = {
        "latencia_s": round(time.perf_counter() - inicio, 2),
        "tentativas_ferramenta": tentativas,
        "degradado": degradou,
        "formato_quebrado": formato_quebrado,
        "problemas": checagem["problemas"],
    }
    return resultado, metricas

print("[done]")

In [ ]:
random.seed(3)
resultado, metricas = responder_robusto("Qual é o prazo para submissão?")
print("RESPOSTA:", resultado.answer)
print("MÉTRICAS:", metricas)

## 8. Variação entre execuções

Temperatura zero reduz a variação, mas não a elimina. Executamos o conjunto três vezes e olhamos
para a faixa, não para a melhor rodada.

In [ ]:
import pandas as pd

REPETICOES = 3
linhas = []

for rodada in range(1, REPETICOES + 1):
    random.seed(100 + rodada)
    for caso in test_cases:
        r, m = responder_robusto(caso["pergunta"])
        a = avaliar(caso, r)
        linhas.append({"rodada": rodada, "id": caso["id"], "tipo": caso["tipo"],
                       "aprovado": a["aprovado"], "degradado": m["degradado"],
                       "problemas": len(m["problemas"]),
                       "latencia_s": m["latencia_s"], "resposta": r.answer})
    print(f"rodada {rodada} concluída")

execucoes = pd.DataFrame(linhas)
execucoes.head()

In [ ]:
por_rodada = (execucoes[execucoes["aprovado"].notna()]
              .groupby("rodada")["aprovado"]
              .agg(acertos="sum", total="count"))
por_rodada["taxa"] = (por_rodada["acertos"] / por_rodada["total"]).round(2)
print(por_rodada, "\n")

faixa = (por_rodada["taxa"].min(), por_rodada["taxa"].max())
print(f"faixa observada: {faixa[0]:.0%} a {faixa[1]:.0%}")

instaveis = (execucoes[execucoes["aprovado"].notna()]
             .groupby("id")["aprovado"]
             .nunique()
             .pipe(lambda s: s[s > 1].index.tolist()))
print("casos que mudaram de resultado entre execuções:", instaveis or "nenhum")

Os casos instáveis marcam a fronteira da capacidade do sistema. Eles merecem mais atenção do que os
que sempre passam ou sempre falham.

In [ ]:
CONFIABILIDADE = {
    "repeticoes": REPETICOES,
    "taxa_min": float(por_rodada["taxa"].min()),
    "taxa_max": float(por_rodada["taxa"].max()),
    "casos_instaveis": instaveis,
    "execucoes_degradadas": int(execucoes["degradado"].sum()),
    "problemas_detectados": int(execucoes["problemas"].sum()),
    "falha_prob_simulada": FALHA_PROB,
}

with open("v4_confiabilidade.json", "w", encoding="utf-8") as f:
    json.dump({"run": RUN_INFO, "confiabilidade": CONFIABILIDADE,
               "execucoes": linhas}, f, ensure_ascii=False, indent=2, default=str)

CONFIABILIDADE

## 9. Discussão

1. Quantas chamadas extras a repetição custou, e o ganho compensou?
2. Alguma resposta degradada passou sem se identificar como degradada?
3. Qual verificação detectou mais problemas?
4. A faixa entre rodadas é maior ou menor que a diferença entre as suas versões?
5. Em que ponto do fluxo você exigiria confirmação humana, e por quê?

## 10. Exercício

1. Escolha a integração mais frágil do seu sistema e torne-a instável de propósito.
2. Implemente repetição com teto, tempo limite e uma degradação que se declare.
3. Acrescente ao menos uma verificação que converta falha silenciosa em ruidosa.
4. Execute o conjunto congelado três vezes e relate a faixa, não a melhor rodada.
5. Liste os casos instáveis e explique o que os torna instáveis.